# Séance 2 — Nettoyage, Enrichissement & Feature Engineering
**Projet : AniData Lab — Sakura Analytics**

Objectif : Nettoyer le dataset brut et produire un dataset **gold** prêt pour ELK.

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/raw/'
GOLD_PATH = '../data/gold/'

## 1. Chargement des données brutes

In [ ]:
anime_df = pd.read_csv(RAW_PATH + 'anime.csv')
synopsis_df = pd.read_csv(RAW_PATH + 'anime_with_synopsis.csv')
print(f'Chargé : {len(anime_df):,} animes')

## 2. Nettoyage — Suppression des doublons

In [ ]:
before = len(anime_df)
anime_df = anime_df.drop_duplicates(subset=['MAL_ID'])
print(f'Doublons supprimés : {before - len(anime_df)}')

## 3. Nettoyage — Conversion des types

In [ ]:
# Remplacer 'Unknown' par NaN
anime_df.replace('Unknown', np.nan, inplace=True)

# Convertir en numérique toutes les colonnes pertinentes
numeric_cols = ['Score', 'Episodes', 'Members', 'Favorites', 'Ranked', 'Popularity',
                'Watching', 'Completed', 'Dropped', 'Plan to Watch',
                'Score-10','Score-9','Score-8','Score-7','Score-6',
                'Score-5','Score-4','Score-3','Score-2','Score-1']
for col in numeric_cols:
    if col in anime_df.columns:
        anime_df[col] = pd.to_numeric(anime_df[col], errors='coerce')

# Calculer 'Scored By' = somme des votes par note (Score-1 à Score-10)
score_cols = [f'Score-{i}' for i in range(1, 11) if f'Score-{i}' in anime_df.columns]
anime_df['Scored By'] = anime_df[score_cols].sum(axis=1)

print('Types après conversion :')
print(anime_df[['Score','Episodes','Members','Favorites','Scored By']].dtypes)

## 4. Nettoyage — Genres et Studios (multi-valeurs)

In [ ]:
# Parser les listes de genres (ex: 'Action, Adventure, Comedy')
def parse_list_col(val):
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).split(',') if x.strip()]

anime_df['genres_list'] = anime_df['Genres'].apply(parse_list_col)
anime_df['studios_list'] = anime_df['Studios'].apply(parse_list_col)
anime_df['producers_list'] = anime_df['Producers'].apply(parse_list_col)

print('Top 10 genres :')
from collections import Counter
all_genres = [g for sublist in anime_df['genres_list'] for g in sublist]
print(Counter(all_genres).most_common(10))

## 5. Feature Engineering — Métriques métier

In [ ]:
# Score de popularité pondéré (Bayesian average)
# Formule : (v / (v + m)) * R + (m / (v + m)) * C
# v = nombre de votes, m = seuil minimum (5000), R = score moyen, C = moyenne globale
C = anime_df['Score'].mean()  # moyenne globale
m = 5000  # seuil minimum de votes

anime_df['weighted_score'] = (
    (anime_df['Scored By'] / (anime_df['Scored By'] + m)) * anime_df['Score'] +
    (m / (anime_df['Scored By'] + m)) * C
).round(4)

print(f'Moyenne globale C = {C:.2f}')
print('Top 5 par weighted_score :')
print(anime_df.nlargest(5, 'weighted_score')[['Name', 'Score', 'weighted_score', 'Members']])

In [ ]:
# Ratio watching/completed (engagement)
anime_df['Watching'] = pd.to_numeric(anime_df.get('Watching', 0), errors='coerce').fillna(0)
anime_df['Completed'] = pd.to_numeric(anime_df.get('Completed', 0), errors='coerce').fillna(0)
anime_df['Dropped'] = pd.to_numeric(anime_df.get('Dropped', 0), errors='coerce').fillna(0)

total_interactions = anime_df['Watching'] + anime_df['Completed'] + anime_df['Dropped']
anime_df['completion_rate'] = (
    anime_df['Completed'] / total_interactions.replace(0, np.nan)
).round(4)

anime_df['drop_rate'] = (
    anime_df['Dropped'] / total_interactions.replace(0, np.nan)
).round(4)

print('Completion rate — stats :')
print(anime_df['completion_rate'].describe())

In [ ]:
# Classification des studios (grand studio vs indépendant)
TOP_STUDIOS = ['Toei Animation', 'Sunrise', 'J.C.Staff', 'Madhouse', 'Bones',
               'A-1 Pictures', 'Production I.G', 'Kyoto Animation', 'Gainax', 'Shaft']

def classify_studio(studios):
    if not studios:
        return 'Inconnu'
    for s in studios:
        if s in TOP_STUDIOS:
            return 'Grand Studio'
    return 'Indépendant'

anime_df['studio_category'] = anime_df['studios_list'].apply(classify_studio)
print('Répartition des catégories de studios :')
print(anime_df['studio_category'].value_counts())

## 6. Fusion avec les synopsis

In [ ]:
# Jointure sur MAL_ID
synopsis_df = synopsis_df[['MAL_ID', 'sypnopsis']].rename(columns={'sypnopsis': 'synopsis'})
gold_df = anime_df.merge(synopsis_df, on='MAL_ID', how='left')
print(f'Dataset gold : {len(gold_df):,} animes, {gold_df.shape[1]} colonnes')

## 7. Export du dataset gold

In [ ]:
import os
os.makedirs(GOLD_PATH, exist_ok=True)

# Colonnes finales pour le gold
gold_cols = [
    'MAL_ID', 'Name', 'Score', 'weighted_score', 'Genres', 'genres_list',
    'Type', 'Episodes', 'Status', 'Studios', 'studios_list', 'studio_category',
    'Members', 'Favorites', 'Scored By', 'Rank', 'Popularity',
    'Watching', 'Completed', 'Dropped', 'completion_rate', 'drop_rate',
    'synopsis'
]
gold_cols = [c for c in gold_cols if c in gold_df.columns]
export_df = gold_df[gold_cols].copy()

# Export CSV
export_df.to_csv(GOLD_PATH + 'anime_gold.csv', index=False, encoding='utf-8')
print(f'CSV exporté : {GOLD_PATH}anime_gold.csv')

# Export JSON (pour Elasticsearch)
# Convertir les listes en vraies listes JSON
json_df = export_df.copy()
json_df.to_json(GOLD_PATH + 'anime_gold.json', orient='records', force_ascii=False, indent=2)
print(f'JSON exporté : {GOLD_PATH}anime_gold.json')

print(f'\nDataset gold prêt : {len(export_df):,} animes')